# 01 — Data Acquisition and Cleaning

This notebook loads the original 45S5 bioactive glass scaffold dataset, inspects its structure, cleans column names and data types, checks data quality, defines the regression target, and saves an analysis-ready dataset for later notebooks.

In [1]:
## 1. Imports and Project Paths
from pathlib import Path

import numpy as np
import pandas as pd

print("✓ Imports successful")
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")

✓ Imports successful
NumPy version: 2.4.6
pandas version: 3.0.3


In [2]:
## 2. Define Project Paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_PATH = DATA_DIR / "DJ-EMA6938-rawdata.xlsx"
CLEAN_DATA_PATH = DATA_DIR / "scaffold_data_clean.csv"

print("✓ Project paths defined")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Raw data file exists: {RAW_DATA_PATH.exists()}")

✓ Project paths defined
Notebook directory: /Users/DrewJohnson/Desktop/porous-bioglass-strength-modeling/notebooks
Project root: /Users/DrewJohnson/Desktop/porous-bioglass-strength-modeling
Raw data path: /Users/DrewJohnson/Desktop/porous-bioglass-strength-modeling/data/DJ-EMA6938-rawdata.xlsx
Raw data file exists: True


In [3]:
## 3. Inspect the Excel Workbook
excel_file = pd.ExcelFile(RAW_DATA_PATH)

print("✓ Excel workbook opened successfully")
print("Available sheet names:")
for sheet_name in excel_file.sheet_names:
    print(f"  - {sheet_name}")

✓ Excel workbook opened successfully
Available sheet names:
  - Sheet1


In [4]:
## 4. Load and Preview the Raw 
# Load the workbook using the row that contains the variable names
raw_with_units = pd.read_excel(
    RAW_DATA_PATH,
    sheet_name="Sheet1",
    header=1
)

# Remove completely empty columns
raw_with_units = raw_with_units.dropna(axis=1, how="all")

# Remove the units row so every remaining row is a real sample
scaffold_data = raw_with_units.iloc[1:].reset_index(drop=True).copy()

# Assign clear, final column names with units included
final_column_names = [
    "Sample ID",
    "Group ID",
    "Size",
    "Date Tested",
    "Binder Composition",
    "Feed Mix (g Bioglass : g PEG6000)",
    "Particle Size (microns)",
    "Sintering Temperature (°C)",
    "Sintering Time (hours)",
    "Final Volume (cm^3)",
    "Final Mass (g)",
    "Volume Loss (cm^3)",
    "Volume Loss (%)",
    "Mass Loss (g)",
    "Mass Loss (%)",
    "Compressive Load (N)",
    "Cross-Sectional Area (mm^2)",
    "Maximum Stress (MPa)",
    "Porosity (%)"
]

if scaffold_data.shape[1] != len(final_column_names):
    raise ValueError(
        f"Expected {len(final_column_names)} columns, "
        f"but found {scaffold_data.shape[1]}."
    )

scaffold_data.columns = final_column_names

print("✓ Dataset loaded with correct headers and units")
print(f"Dataset shape: {scaffold_data.shape}")
display(scaffold_data.head())

✓ Dataset loaded with correct headers and units
Dataset shape: (256, 19)


,Sample ID,Group ID,Size,Date Tested,Binder Composition,Feed Mix (g Bioglass : g PEG6000),Particle Size (microns),Sintering Temperature (°C),Sintering Time (hours),Final Volume (cm^3),Final Mass (g),Volume Loss (cm^3),Volume Loss (%),Mass Loss (g),Mass Loss (%),Compressive Load (N),Cross-Sectional Area (mm^2),Maximum Stress (MPa),Porosity (%)
0,1.0,4225:G1<90um,Evans 6mm,2025-04-22,PVA,100:20,<90,930,6,2.02,2.861,-0.3,-12.931034,-0.749,-20.747922,1819.25,169,10.764793,47.49
1,2.0,4225:G1<90um,Evans 6mm,2025-04-22,PVA,100:20,<90,930,6,2.03,2.901,-0.32,-13.617021,-0.769,-20.953678,1930.66,169,11.424024,47.05
2,3.0,4225:G1<90um,Evans 6mm,2025-04-22,PVA,100:20,<90,930,6,2.03,2.877,-0.33,-13.983051,-0.753,-20.743802,1650.06,169,9.763669,47.38
3,4.0,4225:G1<90um,Evans 6mm,2025-04-22,PVA,100:20,<90,930,6,2.03,2.868,-0.34,-14.345992,-0.752,-20.773481,1592.2,169,9.421302,47.61
4,5.0,4225:G1<90um,Evans 6mm,2025-04-22,PVA,100:20,<90,930,6,2.03,2.864,-0.32,-13.617021,-0.756,-20.883978,1570.2,169,9.291124,47.84


In [5]:
## 6. Inspect Data Types
print("Column data types:\n")
print(scaffold_data.dtypes)

Column data types:

Sample ID                                   float64
Group ID                                        str
Size                                            str
Date Tested                          datetime64[us]
Binder Composition                              str
Feed Mix (g Bioglass : g PEG6000)               str
Particle Size (microns)                         str
Sintering Temperature (°C)                      str
Sintering Time (hours)                          str
Final Volume (cm^3)                          object
Final Mass (g)                               object
Volume Loss (cm^3)                           object
Volume Loss (%)                              object
Mass Loss (g)                                object
Mass Loss (%)                                object
Compressive Load (N)                         object
Cross-Sectional Area (mm^2)                  object
Maximum Stress (MPa)                         object
Porosity (%)                                

In [6]:
## 7. Convert Measurement Columns to Numeric Types
numeric_columns = [
    "Sample ID",
    "Sintering Temperature (°C)",
    "Sintering Time (hours)",
    "Final Volume (cm^3)",
    "Final Mass (g)",
    "Volume Loss (cm^3)",
    "Volume Loss (%)",
    "Mass Loss (g)",
    "Mass Loss (%)",
    "Compressive Load (N)",
    "Cross-Sectional Area (mm^2)",
    "Maximum Stress (MPa)",
    "Porosity (%)"
]

for column in numeric_columns:
    scaffold_data[column] = pd.to_numeric(
        scaffold_data[column],
        errors="coerce"
    )

scaffold_data["Sample ID"] = scaffold_data["Sample ID"].astype("Int64")

print("✓ Numeric columns converted successfully")
print(scaffold_data.dtypes)

✓ Numeric columns converted successfully
Sample ID                                     Int64
Group ID                                        str
Size                                            str
Date Tested                          datetime64[us]
Binder Composition                              str
Feed Mix (g Bioglass : g PEG6000)               str
Particle Size (microns)                         str
Sintering Temperature (°C)                    int64
Sintering Time (hours)                        int64
Final Volume (cm^3)                         float64
Final Mass (g)                              float64
Volume Loss (cm^3)                          float64
Volume Loss (%)                             float64
Mass Loss (g)                               float64
Mass Loss (%)                               float64
Compressive Load (N)                        float64
Cross-Sectional Area (mm^2)                 float64
Maximum Stress (MPa)                        float64
Porosity (%)           

In [7]:
## 8. Check Missing Values
missing_summary = (
    scaffold_data.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="Missing Count")
)

missing_summary["Missing (%)"] = (
    missing_summary["Missing Count"] / len(scaffold_data) * 100
).round(2)

print("✓ Missing-value summary created")
display(missing_summary)

✓ Missing-value summary created


,Missing Count,Missing (%)
Sample ID,0,0.0
Final Mass (g),0,0.0
Maximum Stress (MPa),0,0.0
Cross-Sectional Area (mm^2),0,0.0
Compressive Load (N),0,0.0
Mass Loss (%),0,0.0
Mass Loss (g),0,0.0
Volume Loss (%),0,0.0
Volume Loss (cm^3),0,0.0
Final Volume (cm^3),0,0.0


In [8]:
## 9. Check Duplicate Records
duplicate_rows = scaffold_data.duplicated().sum()
duplicate_sample_ids = scaffold_data["Sample ID"].duplicated().sum()

print(f"Duplicate full rows: {duplicate_rows}")
print(f"Duplicate Sample IDs: {duplicate_sample_ids}")

Duplicate full rows: 0
Duplicate Sample IDs: 0


In [9]:
## 10. Define the Target and Save the Cleaned Dataset
TARGET_COLUMN = "Maximum Stress (MPa)"

if TARGET_COLUMN not in scaffold_data.columns:
    raise KeyError(f"Target column not found: {TARGET_COLUMN}")

scaffold_data.to_csv(CLEAN_DATA_PATH, index=False)

print("✓ Target column confirmed")
print(f"Target: {TARGET_COLUMN}")
print(f"✓ Cleaned dataset saved to: {CLEAN_DATA_PATH}")
print(f"Saved dataset shape: {scaffold_data.shape}")

✓ Target column confirmed
Target: Maximum Stress (MPa)
✓ Cleaned dataset saved to: /Users/DrewJohnson/Desktop/porous-bioglass-strength-modeling/data/scaffold_data_clean.csv
Saved dataset shape: (256, 19)


In [10]:
## 11. Notebook Summary

# The original Excel workbook was loaded using a relative project path. Empty columns and the units row were removed, clear headers with units were assigned, measurement columns were converted to numeric types, and the dataset was checked for missing values and duplicate records.

# The regression target is `Maximum Stress (MPa)`. The cleaned dataset contains 256 samples and 19 columns and was saved as `data/scaffold_data_clean.csv` for use in the exploratory data analysis notebook.